In [ ]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split
from transformers import DistilBertModel, DistilBertTokenizerFast
from torch.utils.data import DataLoader
from torch.optim import AdamW

# PREP THE DATASET

In [2]:
!wget https://raw.githubusercontent.com/kyuz0/llm-chronicles/main/datasets/restaurant_reviews.csv

--2026-08-21 20:14:49--  https://raw.githubusercontent.com/kyuz0/llm-chronicles/main/datasets/restaurant_reviews.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2861025 (2.7M) [text/plain]
Saving to: ‘restaurant_reviews.csv’

restaurant_reviews. 100%[===================>]   2.73M  --.-KB/s    in 0.04s   

2026-08-21 20:14:49 (68.1 MB/s) - ‘restaurant_reviews.csv’ saved [2861025/2861025]



In [3]:
df=pd.read_csv('restaurant_reviews.csv')
df.head()

,Review,Rating
0,The ambience was good food was quite good . ha...,positive
1,Ambience is too good for a pleasant evening. S...,positive
2,A must try.. great food great ambience. Thnx f...,positive
3,Soumen das and Arun was a great guy. Only beca...,positive
4,Food is good.we ordered Kodi drumsticks and ba...,positive


In [4]:
df.Rating.value_counts()

,count
Rating,
positive,6331
negative,2428
neutral,1192


In [5]:
sentiment_mapping={"positive":0, "negative":1,"neutral":3}

In [6]:
df.Rating=df.Rating.map(sentiment_mapping)

In [7]:
# Display the first few rows of the dataframe
print(df.head())

# Display statistics about the dataset
print("\nDataset Statistics:")
print(df['Rating'].value_counts())

                                              Review  Rating
0  The ambience was good food was quite good . ha...       0
1  Ambience is too good for a pleasant evening. S...       0
2  A must try.. great food great ambience. Thnx f...       0
3  Soumen das and Arun was a great guy. Only beca...       0
4  Food is good.we ordered Kodi drumsticks and ba...       0

Dataset Statistics:
Rating
0    6331
1    2428
3    1192
Name: count, dtype: int64


In [8]:
class CustomDatset(Dataset):
  def __init__(self, csv, tokenizer, max_length):
        # Reset index to guarantee contiguous row ordering
        self.dataset = pd.read_csv(csv).reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        # Correcting sentiment_mapping to match the global mapping (neutral: 3)
        self.sentiment_mapping = {"positive": 0, "negative": 1, "neutral": 2}


  def __len__(self):
      return len(self.dataset)

  def __getitem__(self, idx):
    # Ensure review_text is a string
        review_text = str(self.dataset.loc[idx, 'Review'])

        # Normalize text casing to match dictionary keys and map to labels
        sentiment = str(self.dataset.loc[idx, 'Rating']).strip().lower()
        labels = self.sentiment_mapping[sentiment]

        encoding = self.tokenizer(
            review_text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'review_text': review_text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.long)
          }

In [9]:
from transformers import AutoTokenizer
tokenizer_distillbert=AutoTokenizer.from_pretrained('distilbert-base-uncased', use_fast=True)
review_dataset=CustomDatset('restaurant_reviews.csv', tokenizer_distillbert, 512)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
review_dataset[0]

{'review_text': 'The ambience was good food was quite good . had Saturday lunch which was cost effective . Good place for a sate brunch. One can also chill with friends and or parents. Waiter Soumen Das was really courteous and helpful.',
 'input_ids': tensor([  101,  1996,  2572, 11283,  5897,  2001,  2204,  2833,  2001,  3243,
          2204,  1012,  2018,  5095,  6265,  2029,  2001,  3465,  4621,  1012,
          2204,  2173,  2005,  1037,  2938,  2063,  7987,  4609,  2818,  1012,
          2028,  2064,  2036, 10720,  2007,  2814,  1998,  2030,  3008,  1012,
         15610,  2061, 27417,  8695,  2001,  2428,  2457, 14769,  1998, 14044,
          1012,   102,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0, 

In [12]:
tokenizer_distillbert.decode(review_dataset[0]['input_ids'])

'[CLS] the ambience was good food was quite good. had saturday lunch which was cost effective. good place for a sate brunch. one can also chill with friends and or parents. waiter soumen das was really courteous and helpful. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [

In [13]:
from torch.utils.data import DataLoader, random_split
train_size = int(0.8 * len(review_dataset))
val_size = len(review_dataset) - train_size
train_dataset, test_dataset = random_split(review_dataset, [train_size, val_size])
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [15]:
len(train_dataloader),len(test_dataloader)

(995, 249)

# Model

In [17]:
class CustomDistilBertForSequenceClassification(nn.Module):
  def __init__(self,num_labels=3):
    super().__init__()
    self.distilbert=DistilBertModel.from_pretrained('distilbert-base-uncased')
    self.pre_classifier=nn.Linear(768,768)
    self.dropout=nn.Dropout(0.1)
    self.classifier=nn.Linear(768,num_labels)


  def forward(self, input_ids,attention_mask):
    distilbert_output=self.distilbert(input_ids=input_ids,attention_mask=attention_mask)
    hidden_state=distilbert_output[0]
    pooler=hidden_state[:,0]
    pooled_output = self.pre_classifier(pooled_output)
    pooled_output = nn.ReLU()(pooled_output)
    pooled_output = self.dropout(pooled_output) # regularization
    logits = self.classifier(pooled_output)
    return logits



In [18]:
model = CustomDistilBertForSequenceClassification()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
print(model.distilbert)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L